# ViLT + CLIP Cross-Attention Fusion — Fake News Detection
## **10-Fold Cross-Validation** | FakeNewsNet | Kaggle-Ready

**Architecture:**
```
      ViLT Encoder              CLIP Encoders
           ↓                         ↓
    [Joint V+T Embed]        [Image Embed | Text Embed]
           ↓                         ↓
    ─────────────────────────────────────────
    │     Cross-Attention Fusion Network    │
    │  ViLT↔CLIP-Image | ViLT↔CLIP-Text   │
    │  CLIP-Image↔CLIP-Text               │
    │  + Metadata Conditioning             │
    ─────────────────────────────────────────
                    ↓
           Deep Classifier → [Real | Fake]
```

**Changes from 5-fold version:**
- ✅ `n_splits = 10` (10-fold stratified CV)
- ✅ Results table printed after every fold + full summary at the end
- ✅ CSV saved to `/kaggle/working/` after all folds
- ✅ Kaggle paths, checkpointing, GPU AMP — all preserved


In [18]:
# Install dependencies
!pip install -q transformers datasets pillow scikit-learn tqdm


In [19]:
"""
ViLT + CLIP Dual-Encoder Fusion — FakeNewsNet Binary Classification
10-Fold Cross-Validation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Models : ViLT (dandelin/vilt-b32-mlm) + CLIP (openai/clip-vit-base-patch32)
Dataset: rickstello/FakeNewsNet  (train split)
Label  : 0 = real, 1 = fake
CV     : 10-fold stratified  (changed from 5-fold)
Target : ≥ 89% accuracy
"""

from __future__ import annotations

import csv
import io
import os
import random
import math
import warnings
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Any

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    ViltProcessor, ViltModel,
    CLIPProcessor, CLIPModel,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✓ Imports OK")


✓ Imports OK


In [20]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIG  —  only n_splits changed from 5 → 10
# ══════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # Models
    vilt_model_name: str = "dandelin/vilt-b32-mlm"
    clip_model_name: str = "openai/clip-vit-base-patch32"
    dataset_name:    str = "rickstello/FakeNewsNet"
    dataset_split:   str = "train"
    max_samples:     int | None = None
    num_classes:     int = 2

    # Training
    batch_size:        int   = 12
    epochs:            int   = 8
    lr_head:           float = 3e-4
    lr_backbone:       float = 2e-6
    weight_decay:      float = 0.05
    grad_accumulation: int   = 3
    max_grad_norm:     float = 1.0
    warmup_ratio:      float = 0.15

    # Staged fine-tuning
    freeze_epochs:        int = 2
    unfreeze_last_layers: int = 2

    # Data
    n_splits:         int   = 10        # ← CHANGED from 5 to 10
    num_workers:      int   = 2
    image_timeout_s:  float = 10.0
    vilt_image_size:  int   = 384
    clip_image_size:  int   = 224

    # ViLT hard limit: max_position_embeddings = 40.
    # Feeding sequences > 40 raises RuntimeError on token_type_ids buffer.
    vilt_text_max_length: int = 40   # DO NOT raise above 40
    clip_text_max_length: int = 40   # CLIP supports 77; keep aligned
    text_max_length:      int = 40   # legacy alias

    train_frac: float = 0.8
    val_frac:   float = 0.1
    test_frac:  float = 0.1
    require_image_source: bool = False

    # Architecture
    vilt_embed_dim:  int | None = None
    clip_embed_dim:  int | None = None
    fusion_dim:      int   = 256
    n_attn_heads:    int   = 4
    head_dropout:    float = 0.40
    label_smoothing: float = 0.05

    # Metadata
    use_metadata:  bool = True
    metadata_dim:  int  = 64
    max_domains:   int  = 500

    # TF-IDF ensemble
    use_tfidf_ensemble: bool = True
    tfidf_max_features: int  = 80_000
    tfidf_min_df:       int  = 2
    tfidf_ngram_max:    int  = 3

    # Early stopping
    early_stopping_patience: int = 3

    # Kaggle paths
    results_dir:          str  = "/kaggle/working"
    checkpoint_dir:       str  = "/kaggle/working/checkpoints"
    results_file_prefix:  str  = "vilt_clip_fusion_10fold"
    save_checkpoints:     bool = True
    force_cpu:            bool = False

    # AMP
    use_amp: bool = True


config = Config()
print(f"CV folds : {config.n_splits}")
print(f"Epochs   : {config.epochs}")
print(f"Batch    : {config.batch_size}")
print(f"Results  : {config.results_dir}")


CV folds : 10
Epochs   : 8
Batch    : 12
Results  : /kaggle/working


In [21]:
# ══════════════════════════════════════════════════════════════════════
#  DEVICE / AMP
# ══════════════════════════════════════════════════════════════════════
def resolve_device(*, force_cpu: bool = False) -> torch.device:
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, _ = torch.cuda.get_device_capability(0)
        if major < 7:
            print(f"GPU compute capability {major} < 7; falling back to CPU.")
            return torch.device("cpu")
    except Exception as e:
        print(f"CUDA check failed: {e}; using CPU.")
        return torch.device("cpu")
    return torch.device("cuda")


def build_amp_helpers(use_cuda: bool):
    if not use_cuda:
        return (lambda: nullcontext()), torch.cuda.amp.GradScaler(enabled=False)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        return (lambda: torch.amp.autocast("cuda")), scaler
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        return (lambda: torch.cuda.amp.autocast()), scaler


In [22]:
# ══════════════════════════════════════════════════════════════════════
#  TEXT / IMAGE / METADATA HELPERS
# ══════════════════════════════════════════════════════════════════════
def _clean(v: object) -> str:
    s = str(v).strip() if v is not None else ""
    return "" if not s or s.lower() in {"null", "none", "nan"} else s


def get_article_text(ex: dict) -> str:
    parts, seen = [], set()
    for k in ("title", "headline", "text", "article", "content", "body"):
        s = _clean(ex.get(k))
        norm = " ".join(s.lower().split())
        if s and norm not in seen:
            parts.append(s)
            seen.add(norm)
    return " ".join(parts)


def extract_metadata_features(ex: dict) -> dict:
    domain = _clean(ex.get("source_domain", "")).lower() or "unknown"
    try:
        tweet_num = float(ex.get("tweet_num", 0) or 0)
    except (TypeError, ValueError):
        tweet_num = 0.0
    tweet_bin    = 0 if tweet_num <= 0 else min(9, int(math.log1p(tweet_num)))
    text         = get_article_text(ex)
    text_len_bin = min(9, int(math.log1p(len(text.split()))))
    has_image    = get_image_url(ex) is not None or decode_hf_image(ex) is not None
    return {
        "domain":        domain,
        "tweet_bin":     tweet_bin,
        "text_len_bin":  text_len_bin,
        "has_image":     int(has_image),
        "tweet_num_log": math.log1p(tweet_num),
    }


def get_metadata_text(ex: dict) -> str:
    text   = get_article_text(ex)
    domain = _clean(ex.get("source_domain"))
    try:
        tweets = float(ex.get("tweet_num", 0) or 0)
    except (TypeError, ValueError):
        tweets = 0.0
    tweet_bucket = f"tweet_bin_{min(9, int(math.log1p(tweets)))}" if tweets > 0 else "tweet_bin_0"
    parts = [text]
    if domain:
        parts += [f"domain_{domain.lower()}", f"domain_tokens_{domain.lower().replace('.', ' ')}"]
    parts.append(tweet_bucket)
    return " ".join(parts)


def get_image_url(ex: dict) -> str | None:
    for k in ("image_url", "top_img_url", "top_img", "image", "img_url",
              "tweet_image_url", "thumbnail"):
        v = ex.get(k)
        if v and isinstance(v, str) and v.strip() and v.lower() not in ("null", "none", ""):
            return v.strip()
    return None


def decode_hf_image(ex: dict) -> Image.Image | None:
    v = ex.get("image")
    if v is None:
        return None
    if isinstance(v, Image.Image):
        return v.convert("RGB")
    if isinstance(v, dict) and v.get("bytes"):
        try:
            return Image.open(io.BytesIO(v["bytes"])).convert("RGB")
        except Exception:
            return None
    return None


def load_rgb_image(ex: dict, target_size: int = 224) -> Image.Image:
    url = get_image_url(ex)
    if url:
        try:
            r = requests.get(url, timeout=config.image_timeout_s, stream=True)
            r.raise_for_status()
            return Image.open(io.BytesIO(r.content)).convert("RGB")
        except Exception:
            pass
    pil = decode_hf_image(ex)
    if pil is not None:
        return pil
    return Image.new("RGB", (target_size, target_size), color=(128, 128, 128))


def get_binary_label_fake_is_one(ex: dict) -> int | None:
    if ex.get("real") is not None:
        try:
            r = int(ex["real"])
            if r in (0, 1):
                return 1 - r
        except (TypeError, ValueError):
            pass
    for k in ("is_fake", "label"):
        v = ex.get(k)
        if v is not None:
            try:
                lab = int(v)
                if lab in (0, 1):
                    return lab
            except (TypeError, ValueError):
                pass
    return None


def row_has_image_source(ex: dict) -> bool:
    return get_image_url(ex) is not None or decode_hf_image(ex) is not None


In [23]:
# ══════════════════════════════════════════════════════════════════════
#  METADATA ENCODER
# ══════════════════════════════════════════════════════════════════════
class MetadataEncoder:
    def __init__(self, max_domains: int = 500):
        self.max_domains  = max_domains
        self.domain_vocab = {}
        self.fitted       = False

    def fit(self, hf_dataset, indices: list[int]):
        domain_counts = Counter()
        for i in indices:
            meta = extract_metadata_features(hf_dataset[i])
            domain_counts[meta["domain"]] += 1
        top_domains = [d for d, _ in domain_counts.most_common(self.max_domains - 1)]
        self.domain_vocab = {d: i for i, d in enumerate(top_domains)}
        self.domain_vocab["<UNK>"] = len(self.domain_vocab)
        self.fitted = True
        print(f"  Metadata encoder: {len(self.domain_vocab)} unique domains")

    def encode(self, meta_dict: dict) -> dict:
        if not self.fitted:
            raise RuntimeError("MetadataEncoder must be fitted first")
        domain_id = self.domain_vocab.get(meta_dict["domain"], self.domain_vocab["<UNK>"])
        return {
            "domain_id":     domain_id,
            "tweet_bin":     meta_dict["tweet_bin"],
            "text_len_bin":  meta_dict["text_len_bin"],
            "has_image":     meta_dict["has_image"],
            "tweet_num_log": meta_dict["tweet_num_log"],
        }

    @property
    def num_domains(self) -> int:
        return len(self.domain_vocab) if self.fitted else 0


In [24]:
# ══════════════════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════════════════
VILT_TEXT_MAX = 40   # hard limit — ViLT max_position_embeddings = 40
CLIP_TEXT_MAX = 40   # CLIP supports 77; keep equal for simplicity


class DualEncoderDataset(Dataset):
    def __init__(self, hf_dataset, vilt_processor, clip_processor,
                 metadata_encoder: MetadataEncoder,
                 indices: list[int] | None = None):
        self.rows             = hf_dataset
        self.vilt_processor   = vilt_processor
        self.clip_processor   = clip_processor
        self.metadata_encoder = metadata_encoder
        self.indices          = indices if indices is not None else list(range(len(hf_dataset)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        i  = self.indices[idx]
        ex = self.rows[i]
        text = get_article_text(ex)
        if not text:
            raise ValueError("empty text")
        lab = get_binary_label_fake_is_one(ex)
        if lab is None:
            raise ValueError("missing label")

        img_vilt = load_rgb_image(ex, config.vilt_image_size)
        img_clip = load_rgb_image(ex, config.clip_image_size)

        vilt_enc = self.vilt_processor(
            images=img_vilt, text=text,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=VILT_TEXT_MAX,
        )
        vilt_item = {f"vilt_{k}": v.squeeze(0) for k, v in vilt_enc.items()}

        clip_enc = self.clip_processor(
            text=text, images=img_clip,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=CLIP_TEXT_MAX,
        )
        clip_item = {f"clip_{k}": v.squeeze(0) for k, v in clip_enc.items()}

        item = {**vilt_item, **clip_item}

        if config.use_metadata:
            meta         = extract_metadata_features(ex)
            encoded_meta = self.metadata_encoder.encode(meta)
            item["metadata"] = {
                "domain_id":    torch.tensor(encoded_meta["domain_id"],    dtype=torch.long),
                "tweet_bin":    torch.tensor(encoded_meta["tweet_bin"],    dtype=torch.long),
                "text_len_bin": torch.tensor(encoded_meta["text_len_bin"], dtype=torch.long),
                "has_image":    torch.tensor(encoded_meta["has_image"],    dtype=torch.float32),
                "tweet_num_log":torch.tensor(encoded_meta["tweet_num_log"],dtype=torch.float32),
            }

        item["labels"] = torch.tensor(lab, dtype=torch.long)
        return item


def collate_fn(batch: list[dict]) -> dict:
    labels   = torch.stack([b["labels"] for b in batch], dim=0)
    metadata = None
    if "metadata" in batch[0]:
        metadata = {
            k: torch.stack([b["metadata"][k] for b in batch], dim=0)
            for k in batch[0]["metadata"].keys()
        }
    skip = {"labels", "metadata"}
    out  = {k: torch.stack([b[k] for b in batch], dim=0)
            for k in batch[0].keys() if k not in skip}
    out["labels"] = labels
    if metadata is not None:
        out["metadata"] = metadata
    return out


In [25]:
# ══════════════════════════════════════════════════════════════════════
#  CROSS-ATTENTION + METADATA MODULES
# ══════════════════════════════════════════════════════════════════════
class CrossAttentionBlock(nn.Module):
    def __init__(self, D: int, n_heads: int, dropout: float = 0.10):
        super().__init__()
        self.attn = nn.MultiheadAttention(D, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(D)
        self.drop = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key_val: torch.Tensor) -> torch.Tensor:
        q = query.unsqueeze(1)
        k = key_val.unsqueeze(1)
        out, _ = self.attn(q, k, k)
        return self.norm(query + self.drop(out.squeeze(1)))


class MetadataEmbedding(nn.Module):
    def __init__(self, num_domains: int, embed_dim: int = 64):
        super().__init__()
        self.domain_emb   = nn.Embedding(num_domains + 1, embed_dim, padding_idx=0)
        self.tweet_emb    = nn.Embedding(10, 8)
        self.textlen_emb  = nn.Embedding(10, 8)
        self.scalar_proj  = nn.Linear(2, 8)
        self.out_dim      = embed_dim + 8 + 8 + 8

    def forward(self, meta: dict) -> torch.Tensor:
        d = self.domain_emb(meta["domain_id"])
        t = self.tweet_emb(meta["tweet_bin"])
        l = self.textlen_emb(meta["text_len_bin"])
        s = self.scalar_proj(
            torch.stack([meta["has_image"], meta["tweet_num_log"]], dim=-1)
        )
        return torch.cat([d, t, l, s], dim=-1)


# ══════════════════════════════════════════════════════════════════════
#  MAIN FUSION MODEL
# ══════════════════════════════════════════════════════════════════════
class ViLTCLIPDualFusion(nn.Module):
    def __init__(self, num_domains: int, num_classes: int = 2):
        super().__init__()
        D = config.fusion_dim
        H = config.n_attn_heads

        self.vilt = ViltModel.from_pretrained(config.vilt_model_name)
        self.clip = CLIPModel.from_pretrained(config.clip_model_name)

        vilt_dim = self.vilt.config.hidden_size
        clip_dim = self.clip.config.projection_dim
        if config.vilt_embed_dim:
            vilt_dim = config.vilt_embed_dim
        if config.clip_embed_dim:
            clip_dim = config.clip_embed_dim

        self.vilt_proj     = nn.Sequential(nn.Linear(vilt_dim, D), nn.LayerNorm(D), nn.GELU())
        self.clip_img_proj = nn.Sequential(nn.Linear(clip_dim, D), nn.LayerNorm(D), nn.GELU())
        self.clip_txt_proj = nn.Sequential(nn.Linear(clip_dim, D), nn.LayerNorm(D), nn.GELU())

        # Cross-attention pairs
        self.ca_vilt2clip_img = CrossAttentionBlock(D, H)
        self.ca_vilt2clip_txt = CrossAttentionBlock(D, H)
        self.ca_clip_img2vilt = CrossAttentionBlock(D, H)
        self.ca_clip_txt2vilt = CrossAttentionBlock(D, H)
        self.ca_clip_img2txt  = CrossAttentionBlock(D, H)
        self.ca_clip_txt2img  = CrossAttentionBlock(D, H)

        # Metadata
        if config.use_metadata:
            self.metadata_embed = MetadataEmbedding(num_domains, config.metadata_dim)
            meta_dim = self.metadata_embed.out_dim
            self.meta_proj        = nn.Sequential(nn.Linear(meta_dim, D), nn.LayerNorm(D), nn.GELU())
            self.ca_meta2vilt     = CrossAttentionBlock(D, H)
            self.ca_meta2clip_img = CrossAttentionBlock(D, H)
            self.ca_meta2clip_txt = CrossAttentionBlock(D, H)
            n_parts = 13   # 3 proj + 1 meta + 9 cross-attn
        else:
            n_parts = 9    # 3 proj + 6 cross-attn

        in_dim = D * n_parts

        self.classifier = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 512),
            nn.GELU(),
            nn.Dropout(config.head_dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(config.head_dropout / 2),
            nn.Linear(128, num_classes),
        )

    def freeze_backbones(self):
        for p in self.vilt.parameters():
            p.requires_grad = False
        for p in self.clip.parameters():
            p.requires_grad = False

    def unfreeze_last_layers(self, n: int = 2):
        for layer in self.vilt.encoder.layer[-n:]:
            for p in layer.parameters():
                p.requires_grad = True
        try:
            for p in self.vilt.layernorm.parameters():
                p.requires_grad = True
        except AttributeError:
            for p in self.vilt.parameters():
                p.requires_grad = True
        try:
            for layer in self.clip.vision_model.encoder.layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.clip.vision_model.post_layernorm.parameters():
                p.requires_grad = True
        except AttributeError:
            pass
        try:
            for layer in self.clip.text_model.encoder.layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.clip.text_model.final_layer_norm.parameters():
                p.requires_grad = True
        except AttributeError:
            pass
        n_vilt = sum(p.numel() for p in self.vilt.parameters() if p.requires_grad)
        n_clip = sum(p.numel() for p in self.clip.parameters() if p.requires_grad)
        print(f"    ViLT unfrozen: {n_vilt/1e6:.1f}M params")
        print(f"    CLIP unfrozen: {n_clip/1e6:.1f}M params")

    def backbone_params_for_optimizer(self, lr: float) -> list[dict]:
        params = ([p for p in self.vilt.parameters() if p.requires_grad] +
                  [p for p in self.clip.parameters() if p.requires_grad])
        return [{"params": params, "lr": lr, "weight_decay": config.weight_decay}]

    def head_params_for_optimizer(self, lr: float) -> list[dict]:
        mods = [self.vilt_proj, self.clip_img_proj, self.clip_txt_proj,
                self.ca_vilt2clip_img, self.ca_vilt2clip_txt,
                self.ca_clip_img2vilt, self.ca_clip_txt2vilt,
                self.ca_clip_img2txt,  self.ca_clip_txt2img,
                self.classifier]
        if config.use_metadata:
            mods += [self.metadata_embed, self.meta_proj,
                     self.ca_meta2vilt, self.ca_meta2clip_img, self.ca_meta2clip_txt]
        return [{"params": [p for m in mods for p in m.parameters()],
                 "lr": lr, "weight_decay": config.weight_decay}]

    def forward(
        self,
        vilt_input_ids:      torch.Tensor,
        vilt_attention_mask: torch.Tensor,
        vilt_pixel_values:   torch.Tensor,
        clip_input_ids:      torch.Tensor,
        clip_pixel_values:   torch.Tensor,
        clip_attention_mask: torch.Tensor,
        vilt_pixel_mask:         torch.Tensor | None = None,
        vilt_token_type_ids:     torch.Tensor | None = None,
        clip_token_type_ids:     torch.Tensor | None = None,
        clip_attention_mask_2:   torch.Tensor | None = None,
        metadata:    dict | None = None,
        labels:      torch.Tensor | None = None,
        class_weights: torch.Tensor | None = None,
        **unused,
    ):
        vilt_kwargs = dict(
            input_ids      = vilt_input_ids,
            attention_mask = vilt_attention_mask,
            pixel_values   = vilt_pixel_values,
            return_dict    = True,
        )
        if vilt_pixel_mask is not None:
            vilt_kwargs["pixel_mask"] = vilt_pixel_mask
        if vilt_token_type_ids is not None:
            vilt_kwargs["token_type_ids"] = vilt_token_type_ids

        vilt_emb = self.vilt(**vilt_kwargs).pooler_output

        clip_out     = self.clip(
            input_ids      = clip_input_ids,
            pixel_values   = clip_pixel_values,
            attention_mask = clip_attention_mask,
            return_dict    = True,
        )
        clip_img_emb = clip_out.image_embeds
        clip_txt_emb = clip_out.text_embeds

        p_vilt     = self.vilt_proj(vilt_emb)
        p_clip_img = self.clip_img_proj(clip_img_emb)
        p_clip_txt = self.clip_txt_proj(clip_txt_emb)

        ca_v2ci  = self.ca_vilt2clip_img(p_vilt,     p_clip_img)
        ca_v2ct  = self.ca_vilt2clip_txt(p_vilt,     p_clip_txt)
        ca_ci2v  = self.ca_clip_img2vilt(p_clip_img, p_vilt)
        ca_ct2v  = self.ca_clip_txt2vilt(p_clip_txt, p_vilt)
        ca_ci2ct = self.ca_clip_img2txt( p_clip_img, p_clip_txt)
        ca_ct2ci = self.ca_clip_txt2img( p_clip_txt, p_clip_img)

        if config.use_metadata and metadata is not None:
            meta_emb = self.metadata_embed(metadata)
            p_meta   = self.meta_proj(meta_emb)
            ca_m2v   = self.ca_meta2vilt(    p_meta, p_vilt)
            ca_m2ci  = self.ca_meta2clip_img(p_meta, p_clip_img)
            ca_m2ct  = self.ca_meta2clip_txt(p_meta, p_clip_txt)
            fused = torch.cat([
                p_vilt, p_clip_img, p_clip_txt, p_meta,
                ca_v2ci, ca_v2ct, ca_ci2v, ca_ct2v, ca_ci2ct, ca_ct2ci,
                ca_m2v, ca_m2ci, ca_m2ct,
            ], dim=-1)
        else:
            fused = torch.cat([
                p_vilt, p_clip_img, p_clip_txt,
                ca_v2ci, ca_v2ct, ca_ci2v, ca_ct2v, ca_ci2ct, ca_ct2ci,
            ], dim=-1)

        logits = self.classifier(fused)
        loss   = None
        if labels is not None:
            loss = F.cross_entropy(
                logits, labels,
                weight=class_weights,
                label_smoothing=config.label_smoothing,
            )
        return {"loss": loss, "logits": logits}


In [26]:
# ══════════════════════════════════════════════════════════════════════
#  METRICS & UTILITIES
# ══════════════════════════════════════════════════════════════════════
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred,    average="macro", zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred,        average="macro", zero_division=0)),
    }


def class_weights_from_labels(labels: np.ndarray, device: torch.device) -> torch.Tensor:
    counts  = np.bincount(labels, minlength=config.num_classes).astype(np.float32)
    weights = counts.sum() / (config.num_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device)


@torch.no_grad()
def collect_probabilities(model, loader, device, autocast_ctx, class_weights=None):
    model.eval()
    all_probs, all_y = [], []
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch  = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                  for k, v in batch.items()}
        if "metadata" in batch:
            batch["metadata"] = {k: v.to(device) for k, v in batch["metadata"].items()}
        with autocast_ctx():
            logits = model(**batch, labels=None)["logits"]
        all_probs.append(F.softmax(logits.float(), dim=-1).cpu().numpy())
        all_y.append(labels.cpu().numpy())
    return np.concatenate(all_y), np.vstack(all_probs)


def probs_to_preds(probs: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    if probs.shape[1] == 2:
        return (probs[:, 1] >= threshold).astype(np.int64)
    return probs.argmax(axis=1)


def tune_binary_blend(y_true, neural_probs, tfidf_probs=None):
    best_weight    = 1.0
    best_threshold = 0.5
    best_metrics   = compute_metrics(y_true, probs_to_preds(neural_probs))
    best_preds     = probs_to_preds(neural_probs)
    weights        = [1.0] if tfidf_probs is None else np.linspace(0.0, 1.0, 21)
    for w in weights:
        blended = neural_probs if tfidf_probs is None else w * neural_probs + (1.0 - w) * tfidf_probs
        for threshold in np.linspace(0.20, 0.80, 61):
            preds   = probs_to_preds(blended, float(threshold))
            metrics = compute_metrics(y_true, preds)
            if (metrics["f1"] > best_metrics["f1"] or
                (abs(metrics["f1"] - best_metrics["f1"]) < 1e-12 and
                 metrics["accuracy"] > best_metrics["accuracy"])):
                best_weight, best_threshold, best_metrics, best_preds = (
                    float(w), float(threshold), metrics, preds)
    return best_weight, best_threshold, best_metrics, best_preds


def build_tfidf_model(hf_dataset, train_abs: list[int]):
    texts  = [get_metadata_text(hf_dataset[i]) for i in train_abs]
    labels = np.array([get_binary_label_fake_is_one(hf_dataset[i]) for i in train_abs], dtype=np.int64)
    vec    = TfidfVectorizer(
        lowercase=True, strip_accents="unicode", sublinear_tf=True,
        min_df=config.tfidf_min_df, max_features=config.tfidf_max_features,
        ngram_range=(1, config.tfidf_ngram_max),
        token_pattern=r"(?u)\b[\w./:-]+\b",
    )
    x_train = vec.fit_transform(texts)
    clf     = LogisticRegression(C=3.0, class_weight="balanced",
                                 max_iter=2000, solver="liblinear", random_state=SEED)
    clf.fit(x_train, labels)
    return vec, clf, accuracy_score(labels, clf.predict(x_train))


def tfidf_predict_proba(vec, clf, hf_dataset, indices: list[int]) -> np.ndarray:
    texts = [get_metadata_text(hf_dataset[i]) for i in indices]
    return clf.predict_proba(vec.transform(texts))


def build_valid_indices(hf_dataset, max_n: int) -> list[int]:
    valid = []
    for i in tqdm(range(min(len(hf_dataset), max_n)), desc="Scanning samples"):
        ex = hf_dataset[i]
        if not get_article_text(ex):
            continue
        if get_binary_label_fake_is_one(ex) is None:
            continue
        if config.require_image_source and not row_has_image_source(ex):
            continue
        valid.append(i)
    return valid


In [27]:
# ══════════════════════════════════════════════════════════════════════
#  TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════
def train_fold(model, train_loader, val_loader, test_loader,
               device, fold_idx, cw,
               val_tfidf_probs=None, test_tfidf_probs=None) -> dict:

    use_cuda     = device.type == "cuda"
    autocast_ctx, scaler = build_amp_helpers(use_cuda and config.use_amp)

    model.freeze_backbones()
    print(f"  Phase 1: backbones frozen for {config.freeze_epochs} epochs")

    param_groups = (model.head_params_for_optimizer(config.lr_head) +
                    model.backbone_params_for_optimizer(config.lr_backbone))
    optimizer    = torch.optim.AdamW(param_groups)

    steps_ep = max(1, len(train_loader) // config.grad_accumulation)
    t_total  = steps_ep * config.epochs
    warmup   = max(1, int(config.warmup_ratio * t_total))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup, num_training_steps=t_total
    )

    best_val_f1  = -1.0
    best_snapshot = None
    no_improve    = 0
    epoch_logs    = []

    for epoch in range(1, config.epochs + 1):
        if epoch == config.freeze_epochs + 1:
            model.unfreeze_last_layers(config.unfreeze_last_layers)
            print(f"\n  Phase 2: last {config.unfreeze_last_layers} layers unfrozen")
            new_bb = model.backbone_params_for_optimizer(config.lr_backbone)
            optimizer.param_groups[-1]["params"] = new_bb[0]["params"]

        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_preds, train_labels_ep = [], []

        pbar = tqdm(train_loader,
                    desc=f"Fold {fold_idx+1:>2}/{config.n_splits} | Ep {epoch:>2}/{config.epochs}",
                    leave=False)
        for step, batch in enumerate(pbar):
            labels = batch.pop("labels").to(device)
            batch  = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                      for k, v in batch.items()}
            if "metadata" in batch:
                batch["metadata"] = {k: v.to(device) for k, v in batch["metadata"].items()}

            with autocast_ctx():
                out  = model(**batch, labels=labels, class_weights=cw)
                loss = out["loss"] / config.grad_accumulation

            if use_cuda:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % config.grad_accumulation == 0 or (step + 1) == len(train_loader):
                if use_cuda:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                if use_cuda:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            train_preds.extend(out["logits"].argmax(-1).detach().cpu().tolist())
            train_labels_ep.extend(labels.cpu().tolist())
            pbar.set_postfix(loss=f"{loss.item()*config.grad_accumulation:.4f}")

        tr_acc = accuracy_score(train_labels_ep, train_preds)
        y_val,  val_neural_probs  = collect_probabilities(model, val_loader,  device, autocast_ctx, cw)
        y_te,   test_neural_probs = collect_probabilities(model, test_loader, device, autocast_ctx, cw)

        blend_w, threshold, m_val, _ = tune_binary_blend(y_val, val_neural_probs, val_tfidf_probs)
        test_blend = (test_neural_probs if test_tfidf_probs is None
                      else blend_w * test_neural_probs + (1.0 - blend_w) * test_tfidf_probs)
        p_te   = probs_to_preds(test_blend, threshold)
        m_test = compute_metrics(y_te, p_te)
        gap    = tr_acc - m_val["accuracy"]

        epoch_logs.append({
            "fold": fold_idx + 1, "epoch": epoch,
            "train_acc": tr_acc, "val": m_val, "test": m_test,
            "gap": gap, "blend_w": blend_w, "threshold": threshold,
        })

        print(
            f"Fold {fold_idx+1:>2}/{config.n_splits} | Ep {epoch:>2}/{config.epochs} | "
            f"TrainAcc={tr_acc:.4f} | "
            f"VAL  acc={m_val['accuracy']:.4f} f1={m_val['f1']:.4f} | "
            f"TEST acc={m_test['accuracy']:.4f} f1={m_test['f1']:.4f} | "
            f"Gap={gap:+.4f} | blend={blend_w:.2f} thr={threshold:.2f}"
        )

        if m_val["f1"] > best_val_f1:
            best_val_f1    = m_val["f1"]
            best_snapshot  = {
                "epoch": epoch, "val": dict(m_val), "test": dict(m_test),
                "gap": gap, "blend_w": blend_w, "threshold": threshold,
            }
            no_improve = 0
            if config.save_checkpoints:
                os.makedirs(config.checkpoint_dir, exist_ok=True)
                torch.save(
                    {"epoch": epoch, "model_state_dict": model.state_dict(),
                     "metrics": best_snapshot},
                    os.path.join(config.checkpoint_dir, f"fold{fold_idx+1}_best.pt"),
                )
        else:
            if epoch > max(config.freeze_epochs,
                           int(config.warmup_ratio * config.epochs) + 1):
                no_improve += 1
            if no_improve >= config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    assert best_snapshot is not None
    bv, bt = best_snapshot["val"], best_snapshot["test"]
    print(
        f"\n--- Fold {fold_idx+1} best (val F1 @ epoch {best_snapshot['epoch']}) ---\n"
        f"  Val : acc={bv['accuracy']:.4f} prec={bv['precision']:.4f} "
        f"rec={bv['recall']:.4f} f1={bv['f1']:.4f}\n"
        f"  Test: acc={bt['accuracy']:.4f} prec={bt['precision']:.4f} "
        f"rec={bt['recall']:.4f} f1={bt['f1']:.4f}\n"
    )
    return {"fold": fold_idx + 1, "best_by_val_f1": best_snapshot, "epoch_logs": epoch_logs}


In [28]:
# ══════════════════════════════════════════════════════════════════════
#  RESULTS TABLE  —  printed after every fold + full summary
# ══════════════════════════════════════════════════════════════════════
def print_fold_table(fold_reports: list[dict], total_folds: int):
    """Print a running results table after each fold completes."""
    header = (f"{'Fold':>6} | {'Ep':>3} | {'VAL Acc':>8} | {'VAL F1':>7} | "
              f"{'TEST Acc':>9} | {'TEST Prec':>10} | {'TEST Rec':>9} | {'TEST F1':>8} | "
              f"{'Gap':>7} | {'≥89%?':>6}")
    sep = "─" * len(header)
    print(f"\n{'═'*len(header)}")
    print(f"  10-Fold CV Results  ({len(fold_reports)}/{total_folds} folds done)")
    print(f"{'═'*len(header)}")
    print(header)
    print(sep)
    for fr in fold_reports:
        b  = fr["best_by_val_f1"]
        bv, bt = b["val"], b["test"]
        ok = "✅" if bt["accuracy"] >= 0.89 else "❌"
        print(
            f"{fr['fold']:>6} | {b['epoch']:>3} | {bv['accuracy']:>8.4f} | {bv['f1']:>7.4f} | "
            f"{bt['accuracy']:>9.4f} | {bt['precision']:>10.4f} | {bt['recall']:>9.4f} | "
            f"{bt['f1']:>8.4f} | {b['gap']:>+7.4f} | {ok:>6}"
        )
    if len(fold_reports) > 1:
        print(sep)
        # Running mean row
        accs  = [fr["best_by_val_f1"]["test"]["accuracy"]  for fr in fold_reports]
        precs = [fr["best_by_val_f1"]["test"]["precision"] for fr in fold_reports]
        recs  = [fr["best_by_val_f1"]["test"]["recall"]    for fr in fold_reports]
        f1s   = [fr["best_by_val_f1"]["test"]["f1"]        for fr in fold_reports]
        gaps  = [fr["best_by_val_f1"]["gap"]               for fr in fold_reports]
        ok    = "✅" if np.mean(accs) >= 0.89 else "❌"
        print(
            f"{'MEAN':>6} | {'':>3} | {'':>8} | {'':>7} | "
            f"{np.mean(accs):>9.4f} | {np.mean(precs):>10.4f} | {np.mean(recs):>9.4f} | "
            f"{np.mean(f1s):>8.4f} | {np.mean(gaps):>+7.4f} | {ok:>6}"
        )
        print(
            f"{'STD':>6} | {'':>3} | {'':>8} | {'':>7} | "
            f"{np.std(accs):>9.4f} | {np.std(precs):>10.4f} | {np.std(recs):>9.4f} | "
            f"{np.std(f1s):>8.4f} | {'':>7} | {'':>6}"
        )
    print(f"{'═'*len(header)}\n")


def save_and_print_results(fold_reports: list[dict]):
    os.makedirs(config.results_dir, exist_ok=True)
    path = os.path.join(config.results_dir, f"{config.results_file_prefix}_results.csv")

    keys = ["fold", "best_epoch",
            "val_accuracy", "val_precision", "val_recall", "val_f1",
            "test_accuracy", "test_precision", "test_recall", "test_f1",
            "overfit_gap", "blend_w", "threshold"]

    best_rows = []
    for fr in fold_reports:
        b = fr["best_by_val_f1"]
        best_rows.append({
            "fold":           fr["fold"],
            "best_epoch":     b["epoch"],
            "val_accuracy":   b["val"]["accuracy"],
            "val_precision":  b["val"]["precision"],
            "val_recall":     b["val"]["recall"],
            "val_f1":         b["val"]["f1"],
            "test_accuracy":  b["test"]["accuracy"],
            "test_precision": b["test"]["precision"],
            "test_recall":    b["test"]["recall"],
            "test_f1":        b["test"]["f1"],
            "overfit_gap":    b.get("gap", 0.0),
            "blend_w":        b.get("blend_w", 1.0),
            "threshold":      b.get("threshold", 0.5),
        })

    mean_row = {"fold": "MEAN", "best_epoch": ""}
    std_row  = {"fold": "STD",  "best_epoch": ""}
    for k in keys[2:]:
        vals = [r[k] for r in best_rows]
        mean_row[k] = round(float(np.mean(vals)), 6)
        std_row[k]  = round(float(np.std(vals)),  6)

    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in best_rows:
            w.writerow(row)
        w.writerow(mean_row)
        w.writerow(std_row)

    # Final pretty table
    print_fold_table(fold_reports, config.n_splits)

    ma = mean_row["test_accuracy"]
    print(f"  Target ≥ 89%: {'✅ MET' if ma >= 0.89 else f'❌ Gap = {0.89 - ma:.4f}'}")
    print(f"  Results CSV  : {path}\n")


In [29]:
# ══════════════════════════════════════════════════════════════════════
#  MAIN  —  10-Fold CV
# ══════════════════════════════════════════════════════════════════════
def main():
    from datasets import load_dataset

    device = resolve_device(force_cpu=config.force_cpu)
    print(f"\n{'='*72}")
    print(f"  ViLT + CLIP Dual Fusion — FakeNewsNet — {config.n_splits}-Fold CV")
    print(f"{'='*72}")
    print(f"  Device  : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == 'cuda' else ""))
    print(f"  ViLT    : {config.vilt_model_name}")
    print(f"  CLIP    : {config.clip_model_name}")
    print(f"  Metadata: {config.use_metadata}")
    print(f"  Epochs  : {config.epochs}  |  CV: {config.n_splits}-fold  |  Batch: {config.batch_size}")
    print(f"{'='*72}\n")

    print("Loading processors…")
    vilt_processor = ViltProcessor.from_pretrained(config.vilt_model_name)
    clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)

    print("\nLoading FakeNewsNet dataset…")
    ds = load_dataset(config.dataset_name, split=config.dataset_split)
    n  = len(ds) if config.max_samples is None else min(config.max_samples, len(ds))
    ds = ds.select(range(n))
    print(f"  Rows loaded: {len(ds)}")

    valid_idx = build_valid_indices(ds, len(ds))
    labels_all = [get_binary_label_fake_is_one(ds[i]) for i in valid_idx]
    y_all      = np.array(labels_all, dtype=np.int64)
    counts     = np.bincount(y_all, minlength=2)
    print(f"  Valid: {len(valid_idx)}/{len(ds)} | real={counts[0]} | fake={counts[1]}\n")

    # 80 / 10 / 10 stratified split
    pos = np.arange(len(valid_idx))
    pos_train, pos_temp, y_tr, y_tmp = train_test_split(
        pos, y_all,
        test_size=(config.val_frac + config.test_frac),
        stratify=y_all, random_state=SEED,
    )
    rel_test = config.test_frac / (config.val_frac + config.test_frac)
    pos_val, pos_test, _, _ = train_test_split(
        pos_temp, y_tmp, test_size=rel_test, stratify=y_tmp, random_state=SEED
    )

    abs_train_pool = [valid_idx[int(i)] for i in pos_train]
    abs_val        = [valid_idx[int(i)] for i in pos_val]
    abs_test       = [valid_idx[int(i)] for i in pos_test]
    y_train_pool   = y_tr

    print(f"Split: train_pool={len(abs_train_pool)} | val={len(abs_val)} | test={len(abs_test)}\n")

    # Fit metadata encoder on entire train pool
    metadata_encoder = MetadataEncoder(max_domains=config.max_domains)
    metadata_encoder.fit(ds, abs_train_pool)

    # Fixed val/test loaders (shared across all folds)
    val_ds  = DualEncoderDataset(ds, vilt_processor, clip_processor, metadata_encoder, abs_val)
    test_ds = DualEncoderDataset(ds, vilt_processor, clip_processor, metadata_encoder, abs_test)
    val_loader  = DataLoader(val_ds,  batch_size=config.batch_size, shuffle=False,
                             num_workers=config.num_workers, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False,
                             num_workers=config.num_workers, collate_fn=collate_fn)

    skf          = StratifiedKFold(n_splits=config.n_splits, shuffle=True, random_state=SEED)
    fold_reports = []

    for fold_idx, (tr_rel, _) in enumerate(skf.split(np.zeros(len(abs_train_pool)), y_train_pool)):
        print(f"\n{'═'*72}")
        print(f"  FOLD {fold_idx+1} / {config.n_splits}")
        print(f"{'═'*72}")

        fold_train_abs = [abs_train_pool[i] for i in tr_rel]
        fold_labels    = np.array(
            [get_binary_label_fake_is_one(ds[i]) for i in fold_train_abs], dtype=np.int64
        )
        cw = class_weights_from_labels(fold_labels, device)
        print(f"  Train: {len(fold_train_abs)} | weights [real={cw[0]:.3f}, fake={cw[1]:.3f}]")

        # TF-IDF ensemble
        val_tfidf_probs = test_tfidf_probs = None
        if config.use_tfidf_ensemble:
            vec, tfidf_clf, tfidf_train_acc = build_tfidf_model(ds, fold_train_abs)
            val_tfidf_probs  = tfidf_predict_proba(vec, tfidf_clf, ds, abs_val)
            test_tfidf_probs = tfidf_predict_proba(vec, tfidf_clf, ds, abs_test)
            tfidf_val_m      = compute_metrics(
                y_all[pos_val], probs_to_preds(val_tfidf_probs))
            print(f"  TF-IDF: train_acc={tfidf_train_acc:.4f} "
                  f"val_acc={tfidf_val_m['accuracy']:.4f} val_f1={tfidf_val_m['f1']:.4f}")

        train_ds     = DualEncoderDataset(ds, vilt_processor, clip_processor,
                                          metadata_encoder, fold_train_abs)
        train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                                  num_workers=config.num_workers, collate_fn=collate_fn)

        model  = ViLTCLIPDualFusion(
            num_domains=metadata_encoder.num_domains,
            num_classes=config.num_classes,
        ).to(device)

        report = train_fold(
            model, train_loader, val_loader, test_loader, device, fold_idx, cw,
            val_tfidf_probs=val_tfidf_probs, test_tfidf_probs=test_tfidf_probs,
        )
        fold_reports.append(report)

        # Print running results table after each fold
        print_fold_table(fold_reports, config.n_splits)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Final summary + CSV ───────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  FINAL SUMMARY — {config.n_splits}-Fold CV")
    print(f"{'='*72}")
    save_and_print_results(fold_reports)


if __name__ == "__main__":
    main()



  ViLT + CLIP Dual Fusion — FakeNewsNet — 10-Fold CV
  Device  : cuda (Tesla T4)
  ViLT    : dandelin/vilt-b32-mlm
  CLIP    : openai/clip-vit-base-patch32
  Metadata: True
  Epochs  : 8  |  CV: 10-fold  |  Batch: 12

Loading processors…

Loading FakeNewsNet dataset…
  Rows loaded: 23196


Scanning samples:   0%|          | 0/23196 [00:00<?, ?it/s]

  Valid: 23196/23196 | real=17441 | fake=5755

Split: train_pool=18556 | val=2320 | test=2320

  Metadata encoder: 500 unique domains

════════════════════════════════════════════════════════════════════════
  FOLD 1 / 10
════════════════════════════════════════════════════════════════════════
  Train: 16700 | weights [real=0.496, fake=1.504]
  TF-IDF: train_acc=0.9642 val_acc=0.8806 val_f1=0.8444


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>self._shutdown_workers()

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    
 self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive(): 
   ^^ ^^^ ^ ^^  ^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^

  File "/usr/lib/python

Fold 10/10 | Ep  7/8 | TrainAcc=0.9750 | VAL  acc=0.9108 f1=0.8774 | TEST acc=0.9082 f1=0.8744 | Gap=+0.0643 | blend=0.35 thr=0.56


Fold 10/10 | Ep  8/8:   0%|          | 0/1392 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
  Exception ignored in:    <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80> 
 ^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    ^if w.is_alive():

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
       assert self._parent_pid == os.getpid(), 'can only test a child process' 
    ^^ ^ ^  ^ ^ ^ ^ ^ ^ ^^^^^Exception 

Fold 10/10 | Ep  8/8 | TrainAcc=0.9811 | VAL  acc=0.9103 f1=0.8773 | TEST acc=0.9073 f1=0.8734 | Gap=+0.0707 | blend=0.35 thr=0.56
  Early stopping at epoch 8

--- Fold 10 best (val F1 @ epoch 5) ---
  Val : acc=0.9125 prec=0.8856 rec=0.8779 f1=0.8816
  Test: acc=0.9082 prec=0.8802 rec=0.8708 f1=0.8753


════════════════════════════════════════════════════════════════════════════════════════════════════
  10-Fold CV Results  (10/10 folds done)
════════════════════════════════════════════════════════════════════════════════════════════════════
  Fold |  Ep |  VAL Acc |  VAL F1 |  TEST Acc |  TEST Prec |  TEST Rec |  TEST F1 |     Gap |  ≥89%?
────────────────────────────────────────────────────────────────────────────────────────────────────
     1 |   8 |   0.9047 |  0.8739 |    0.8996 |     0.8614 |    0.8749 |   0.8678 | +0.0726 |      ✅
     2 |   4 |   0.9116 |  0.8823 |    0.9103 |     0.8789 |    0.8815 |   0.8802 | +0.0082 |      ✅
     3 |   5 |   0.9181 |  0.8874 |    0.9103 |

In [ ]:
# ▶ Run training
main()



  ViLT + CLIP Dual Fusion — FakeNewsNet — 10-Fold CV
  Device  : cuda (Tesla T4)
  ViLT    : dandelin/vilt-b32-mlm
  CLIP    : openai/clip-vit-base-patch32
  Metadata: True
  Epochs  : 8  |  CV: 10-fold  |  Batch: 12

Loading processors…

Loading FakeNewsNet dataset…
  Rows loaded: 23196


Scanning samples:   0%|          | 0/23196 [00:00<?, ?it/s]

  Valid: 23196/23196 | real=17441 | fake=5755

Split: train_pool=18556 | val=2320 | test=2320

  Metadata encoder: 500 unique domains

════════════════════════════════════════════════════════════════════════
  FOLD 1 / 10
════════════════════════════════════════════════════════════════════════
  Train: 16700 | weights [real=0.496, fake=1.504]
  TF-IDF: train_acc=0.9642 val_acc=0.8806 val_f1=0.8444


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Fold  1/10 | Ep  1/8:   0%|          | 0/1392 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>^^
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^  ^^

Fold  1/10 | Ep  1/8 | TrainAcc=0.7720 | VAL  acc=0.9026 f1=0.8626 | TEST acc=0.9013 f1=0.8603 | Gap=-0.1306 | blend=0.35 thr=0.65


Fold  1/10 | Ep  2/8:   0%|          | 0/1392 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
     Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1eb571ee80>^^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^^if w.is_alive():
^  
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
   ^^  ^^ ^ ^^ ^   ^ ^^^^^
^  File "/